# 02 — Series temporales del bootstrap autorizado

**Preguntas:** ¿qué tendencias y asociaciones con el calendario aparecen entre 2013 y 2014? ¿Cómo varían por tienda/familia? ¿Qué dependencia por retardo persiste después de una transformación descriptiva?

Dos años no prueban una estacionalidad anual estable. Las correlaciones y cambios visuales no identifican causas. No se incorporan feriados, petróleo, promociones ni dimensiones restringidas. El calendario se conserva con ausencias NaN, sin interpolar ventas ni inventar ceros.


In [ ]:
from pathlib import Path
import sys
NOTEBOOKS = Path.cwd() if (Path.cwd() / "retailops_eda").is_dir() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOKS))
import pandas as pd
import matplotlib.pyplot as plt
from retailops_eda.data import query, validate_bootstrap
from retailops_eda.analysis import setup, finish, calendar_series, monthly_means, lag_correlations, save_summary, UNIT
setup()
overview = validate_bootstrap()  # trainer, solo lectura, corte fijo y prueba de ausencia futura
print("Bootstrap autorizado: 2013-01-01 → 2014-12-31; filas:", int(overview['records']))


## Tendencia agregada

Medias móviles retrospectivas descriptivas de 7 y 28 días de calendario. Se exigen 5/21 observaciones respectivamente, se conservan NaN de la fuente y no se presenta la media como una feature disponible para un pronóstico. El desplazamiento y la disponibilidad al horizonte se decidirán en F1-04B.


In [ ]:
daily = query('daily')
series = calendar_series(daily)
monthly = monthly_means(query('monthly'))
fig, ax = plt.subplots(figsize=(13,5))
ax.plot(series.index, series/1e3, color='#AAB4C3', linewidth=.8, label='Total diario observado')
ax.plot(series.index, series.rolling(7,min_periods=5).mean()/1e3, color='#315EAF', label='Media retrospectiva 7 días')
ax.plot(series.index, series.rolling(28,min_periods=21).mean()/1e3, color='#BE6822', label='Media retrospectiva 28 días')
ax.set(title='Tendencia descriptiva de ventas agregadas · 2013–2014', xlabel='Fecha', ylabel='Ventas registradas (miles; unidades mixtas)')
ax.legend(); finish(fig,'ts_trend'); plt.show()
yearly = daily.assign(year=daily.date.dt.year).groupby('year').agg(mean_daily=('total_sales','mean'),total=('total_sales','sum'),zero_sales=('zero_sales','sum'),records=('records','sum'))
yearly['zero_rate'] = yearly.zero_sales/yearly.records
display(yearly)


## Asociación semanal y mensual

Para el patrón semanal se promedian totales diarios observados, separados por año para mostrar cambios de nivel. Para el mensual se compara la media por día observado, sin confundir meses de distinta longitud ni dos Navidades ausentes con una caída de ventas cero. Estos son perfiles descriptivos, no efectos causales o estimadores de un patrón futuro fijo.


In [ ]:
week = daily.assign(year=daily.date.dt.year,weekday=daily.date.dt.dayofweek).groupby(['year','weekday']).total_sales.mean().unstack(0)
fig, ax = plt.subplots()
week.div(1e3).plot(ax=ax,marker='o',color=['#315EAF','#BE6822'])
ax.set_xticks(range(7),['Lun','Mar','Mié','Jue','Vie','Sáb','Dom'])
ax.set(title='Perfil semanal de ventas diarias · comparación por año', xlabel='Día de semana', ylabel='Media diaria observada (miles; unidades mixtas)')
ax.legend(title='Año'); finish(fig,'ts_weekly'); plt.show()
annual = monthly.assign(year=monthly.month.dt.year,calendar_month=monthly.month.dt.month).pivot(index='calendar_month',columns='year',values='sales_per_observed_day')
fig, ax = plt.subplots()
annual.div(1e3).plot(ax=ax,marker='o',color=['#315EAF','#BE6822'])
ax.set_xticks(range(1,13),['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
ax.set(title='Perfil mensual: media por día observado · solo dos ciclos anuales', xlabel='Mes de calendario', ylabel='Media diaria observada (miles; unidades mixtas)')
ax.legend(title='Año'); finish(fig,'ts_monthly'); plt.show()


## Series seleccionadas por tienda y familia

Las tres tiendas se seleccionan por identificador (primera, central y última), no por un supuesto tipo o ciudad. Las tres familias de mayor volumen se seleccionan solo del bootstrap. Son ejemplos transparentes, no una muestra representativa; el notebook 01 contiene las distribuciones de todas las tiendas/familias.


In [ ]:
stores = query('by_store').store_nbr.sort_values().tolist()
selected_stores = [int(stores[0]),int(stores[len(stores)//2]),int(stores[-1])]
store_daily = query('store_daily', stores=selected_stores)
selected_families = query('by_family').head(3).family.tolist()
family_daily = query('family_daily', families=selected_families)
fig, axes = plt.subplots(2,1,figsize=(13,8),sharex=True)
for store,group in store_daily.groupby('store_nbr'):
    s = calendar_series(group)
    axes[0].plot(s.index,s.rolling(28,min_periods=21).mean()/1e3,label=f'Tienda {store}')
for family,group in family_daily.groupby('family'):
    s = calendar_series(group)
    axes[1].plot(s.index,s.rolling(28,min_periods=21).mean()/1e3,label=family)
axes[0].set(title='Ejemplos por tienda · media retrospectiva de 28 días',ylabel='Ventas (miles)'); axes[0].legend()
axes[1].set(title='Tres familias de mayor volumen del bootstrap · media retrospectiva 28 días',ylabel='Ventas (miles)',xlabel='Fecha'); axes[1].legend()
finish(fig,'ts_segments'); plt.show()


## Dependencia por retardo y cambios de nivel

Se calcula Pearson entre valores separados por **k días de calendario**, únicamente sobre pares observados. Se muestra la serie agregada y la primera diferencia de log(1+ventas), para advertir cómo tendencia/escala afectan la correlación. No se aplica una ACF de posiciones que comprima los días ausentes, no se imputan ventas, ni se estiman bandas de significación. El número de pares varía con el retardo. La agregación puede ocultar series individuales intermitentes.


In [ ]:
import numpy as np
acf_raw = lag_correlations(series)
acf_diff = lag_correlations(np.log1p(series).diff())
fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True)
for ax,frame,title in [(axes[0],acf_raw,'Serie agregada original'),(axes[1],acf_diff,'Primera diferencia de log(1+ventas)')]:
    ax.vlines(frame.lag_days,0,frame.correlation,color='#315EAF')
    ax.scatter(frame.lag_days,frame.correlation,s=10,color='#315EAF')
    ax.axhline(0,color='black',linewidth=.7)
    for lag in [7,14,28]: ax.axvline(lag,color='#BE6822',alpha=.35,linestyle='--')
    ax.set(title=title,ylabel='Correlación por pares')
axes[1].set_xlabel('Retardo (días de calendario)')
fig.suptitle('Dependencia temporal descriptiva · 2013–2014',y=1.02)
finish(fig,'ts_autocorrelation'); plt.show()
changes = monthly[['month','sales_per_observed_day']].copy()
changes['relative_change'] = changes.sales_per_observed_day.pct_change(fill_method=None)
display(changes.dropna().sort_values('relative_change').iloc[[0,-1]])
display(acf_diff[acf_diff.lag_days.isin([1,7,14,28])])
weekly = daily.assign(weekday=daily.date.dt.dayofweek).groupby('weekday').total_sales.mean()
change_rows = changes.dropna().sort_values('relative_change')
save_summary('time_series',{'yearly':{str(k):{name:float(value) for name,value in row.items()} for k,row in yearly.to_dict('index').items()},'weekday_mean':{str(k):float(v) for k,v in weekly.items()},'monthly_mean':{pd.Timestamp(k).strftime('%Y-%m'):float(v) for k,v in zip(monthly.month,monthly.sales_per_observed_day)},'selected_stores':selected_stores,'selected_families':selected_families,'acf_raw':acf_raw[acf_raw.lag_days.isin([1,7,14,28,56])].to_dict('records'),'acf_log_diff':acf_diff[acf_diff.lag_days.isin([1,7,14,28,56])].to_dict('records'),'largest_decline_month':pd.Timestamp(change_rows.iloc[0].month).strftime('%Y-%m'),'largest_decline':float(change_rows.iloc[0].relative_change),'largest_increase_month':pd.Timestamp(change_rows.iloc[-1].month).strftime('%Y-%m'),'largest_increase':float(change_rows.iloc[-1].relative_change)})


## Interpretación y siguientes hipótesis

Una tendencia creciente, perfil semanal o pico en diciembre puede motivar comparar naive último valor, naive estacional semanal y una media/mediana retrospectiva. No se ha entrenado ni evaluado ningún modelo. Dos ciclos anuales, ceros heterogéneos, composición variable y ausencia de auxiliares limitan las conclusiones. Los cambios mensuales son candidatos de revisión, no cambios de régimen demostrados. Las recomendaciones verificadas de esta ejecución quedan en `docs/EDA_FINDINGS.md`; cualquier GOLD, selección de features y validación al horizonte quedan para F1-04B con autorización.
